<a href="https://colab.research.google.com/github/inmobiliariamatsan17-ui/NUDOS/blob/main/Qwen3_8_27B_Ridge_Colab_Replica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```markdown
# Qwen 3.8B Ridge GGUF with Vision on Colab T4

This notebook sets up a Qwen 3.8B model with vision capabilities using `llama.cpp` on a free Colab T4 GPU, exposing it via an OpenAI-compatible API.
```

```markdown
## 1. Install Dependencies and Check GPU
```

In [ ]:
%%bash
python -m pip install -q pyngrok requests pillow huggingface_hub hf_transfer

# Install cloudflared binary
wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
chmod +x cloudflared-linux-amd64
mv cloudflared-linux-amd64 /usr/local/bin/cloudflared

nvidia-smi

```markdown
## 2. Configuration and Utility Functions

This section defines global variables for model paths, tunnel modes, and includes a function to find an available port for the `llama-server`.
```

In [ ]:
import os, sys
import socket
import subprocess
import time
from google.colab import userdata, drive

# --- Mount Google Drive for persistent storage ---
try:
    print("Mounting Google Drive...")
    drive.mount('/content/drive')
    MODELS_DIR = "/content/drive/MyDrive/llama_models"
    print(f"Google Drive mounted. Models will be saved to: {MODELS_DIR}")
except Exception as e:
    print(f"Failed to mount Google Drive: {e}. Falling back to local temporary storage.")
    MODELS_DIR = "/content/models"

# --- Model Configuration ---
MODEL_REPO = "empero-ai/Qwen3.8-27B-Ridge-GGUF"
MODEL_FILE = "Qwen3.8-27B-Ridge-3.7bpw.gguf"
MMPROJ_FILE = "mmproj-Qwen3.8-27B-BF16.gguf"

LLAMA_SRC = "/content/llama.cpp"
LLAMA_SERVER_BIN = os.path.join(LLAMA_SRC, "build", "bin", "llama-server")

# --- Tunnel Configuration ---
# Recommended mode is Cloudflared; ngrok is fallback.
TUNNEL_MODE = "cloudflared" # Options: "cloudflared", "ngrok"

# NOTE: CF_HOSTNAME and DOMAIN should be your own, do not reuse these examples.
# You need to configure a CNAME record for CF_HOSTNAME to point to your Cloudflare tunnel.
CF_HOSTNAME = "your-custom-cf-hostname.example.com" # Placeholder
DOMAIN = "your-ngrok-domain.ngrok-free.dev" # Placeholder

# Get ngrok token from Colab secrets.
# The exact name might be 'NGROK_AUTHTOKEN' or 'NGROK_TOKEN' based on your secret setup.
try:
    NGROK_TOKEN = userdata.get("NGROK_TOKEN")
except userdata.SecretNotFoundError:
    print("NGROK_TOKEN not found in Colab secrets. ngrok tunneling will not work.")
    NGROK_TOKEN = None


# --- Dynamic Port Finding Utility ---
def find_free_port(start_port=8080, end_port=8090):
    """Finds an available port in the specified range."""
    for port in range(start_port, end_port + 1):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            try:
                s.bind(("127.0.0.1", port))
                return port
            except OSError:
                continue
    raise RuntimeError(f"No free port found between {start_port} and {end_port}")

PORT = find_free_port() # Initial dynamic port assignment
print(f"Using port: {PORT}")

```markdown
## 3. Download Models from Hugging Face

This cell downloads the Qwen 3.8B model and its vision projection (`mmproj`) from Hugging Face Hub. It caches them locally to avoid re-downloading.
```

In [ ]:
from huggingface_hub import hf_hub_download
import os

# Validación de seguridad: Definir variables por si no se ejecutó la celda de configuración
if 'MODELS_DIR' not in locals() and 'MODELS_DIR' not in globals():
    print("Advertencia: Celda de configuración no ejecutada. Usando valores por defecto...")
    MODELS_DIR = "/content/models"
    MODEL_REPO = "empero-ai/Qwen3.8-27B-Ridge-GGUF"
    MODEL_FILE = "Qwen3.8-27B-Ridge-3.7bpw.gguf"
    MMPROJ_FILE = "mmproj-Qwen3.8-27B-BF16.gguf"

os.makedirs(MODELS_DIR, exist_ok=True)

def fetch(filename):
    dest = os.path.join(MODELS_DIR, filename)
    if os.path.exists(dest) and os.path.getsize(dest) > 1_000_000_000:
        print(f"Cached: {filename} ({os.path.getsize(dest) / 1e9:.1f} GB)")
        return dest

    print(f"Downloading: {filename}...")
    path = hf_hub_download(
        repo_id=MODEL_REPO,
        filename=filename,
        local_dir=MODELS_DIR,
    )
    print(f"Downloaded: {filename} ({os.path.getsize(path) / 1e9:.1f} GB)")
    return path

model_path = fetch(MODEL_FILE)
mmproj_path = fetch(MMPROJ_FILE)
print("Model path:", model_path)
print("MMProj path:", mmproj_path)

```markdown
## 4. Download and Prepare `llama-server`

Downloads a pre-compiled `llama-server` binary that supports CUDA 12.8, making it executable. We are assuming a `llama.cpp` GitHub release link for a pre-compiled CUDA binary.
```

In [ ]:
import os

# Validación de seguridad: Definir variables si no se ejecutó la celda de configuración
if 'LLAMA_SRC' not in locals() and 'LLAMA_SRC' not in globals():
    print("Advertencia: Celda de configuración no detectada. Usando rutas por defecto...")
    LLAMA_SRC = "/content/llama.cpp"
    LLAMA_SERVER_BIN = os.path.join(LLAMA_SRC, "build", "bin", "llama-server")

# Clone llama.cpp and build llama-server with CUDA support
print(f"Cloning llama.cpp into {LLAMA_SRC}...")
# Use --depth 1 to only clone the latest commit, saving time and space
!git clone https://github.com/ggerganov/llama.cpp.git {LLAMA_SRC} || echo 'llama.cpp already cloned'

print("Installing build dependencies...")
!apt-get update -qq && apt-get install -y cmake build-essential

print("Building llama-server with CUDA support using CMake...")
%cd {LLAMA_SRC}
!rm -rf build # Clean previous build artifacts
!mkdir -p build
%cd build
!cmake .. -DGGML_CUDA=ON
!cmake --build . --config Release --target llama-server
%cd {LLAMA_SRC} # Return to llama.cpp root for verification

# Verify if llama-server binary exists
if not os.path.exists(LLAMA_SERVER_BIN):
    raise FileNotFoundError(f"llama-server binary not found after compilation in {LLAMA_SERVER_BIN}")

print(f"llama-server binary found at: {LLAMA_SERVER_BIN}")
print("llama-server made executable (implicitly by make).")
%cd /content # Return to default content directory

```markdown
## 5. Start `llama-server`

This cell starts the `llama-server` process using the downloaded model and `mmproj`. It automatically selects an available port and waits for the server to become ready. It also displays VRAM usage.
```

In [ ]:
# Update PORT in case it changed during prior attempts or manual intervention
# (though `find_free_port` is meant to handle this).
# The server logs might indicate the actual port if 8080 was taken.

server_log_file = os.path.join(LLAMA_SRC, "server_output.log")

print(f"Starting llama-server on port {PORT}...")
print(f"Server output will be redirected to: {server_log_file}")

# Command to run llama-server
# The -ngl 99 argument offloads 99 layers to the GPU.
llama_command = [
    LLAMA_SERVER_BIN,
    "-m", model_path,
    "--mmproj", mmproj_path,
    "--port", str(PORT),
    "-ngl", "99", # Offload 99 layers to GPU, adjust based on VRAM
    "-c", "4096", # Context size
    "--host", "0.0.0.0" # Listen on all interfaces
]

# Start the server in a non-blocking way and redirect output
with open(server_log_file, "w") as log_file:
    server_process = subprocess.Popen(llama_command, stdout=log_file, stderr=log_file)

# Wait for the server to be ready and dynamically adjust PORT if it changes
server_ready = False
for _ in range(60): # Wait up to 60 seconds
    with open(server_log_file, "r") as f:
        content = f.read()
        if "HTTP server listening" in content or "llama_new_context_with_f16_kv" in content:
            server_ready = True
            # Check if port changed (e.g., from 8080 to 8081)
            if "NOTE: port" in content and "is already taken" in content:
                import re
                match = re.search(r"using (\d+) instead", content)
                if match:
                    new_port = int(match.group(1))
                    if new_port != PORT:
                        print(f"Server reports using port {new_port} instead of {PORT}.")
                        PORT = new_port
            break
    time.sleep(1)

if server_ready:
    print(f"llama-server READY on port {PORT}.")
    # Display VRAM usage after server starts
    print("\nVRAM Usage after server startup:")
    !nvidia-smi
else:
    print("llama-server did not become ready in time. Check log for errors:")
    with open(server_log_file, "r") as f:
        print(f.read())
    server_process.terminate()
    raise RuntimeError("llama-server failed to start.")

```markdown
## 6. Self-Test OpenAI-Compatible API

This section tests the `llama-server`'s OpenAI-compatible API endpoints for text and vision capabilities.
```

In [ ]:
import io, base64, time, requests
from PIL import Image, ImageDraw

base = f"http://127.0.0.1:{PORT}/v1"

try:
    models = requests.get(base + "/models", timeout=30 ).json()
    MODEL_ID = models["data"][0]["id"] if models.get("data") else "unknown"
    print("Loaded model ID:", MODEL_ID)
except requests.exceptions.RequestException as e:
    print(f"Failed to connect to llama-server at {base}: {e}")
    MODEL_ID = "unknown"

def chat(messages, max_tokens=200, timeout=600):
    t0 = time.time()
    payload = {
        "model": MODEL_ID,
        "messages": messages,
        "max_tokens": max_tokens,
        "stream": False,
    }
    try:
        r = requests.post(
            base + "/chat/completions",
            json=payload,
            timeout=timeout,
        )
        r.raise_for_status() # Raise an exception for HTTP errors (4xx or 5xx)
        j = r.json()
        if "choices" not in j or not j["choices"]:
            raise RuntimeError(f"No choices in API response: {j}")
        return j["choices"][0]["message"]["content"], time.time() - t0
    except requests.exceptions.RequestException as e:
        raise RuntimeError(f"API request failed: {e}\nResponse: {getattr(e.response, 'text', 'No response body')[:500]}")

# --- Text Test ---
print("\n--- Text Test ---")
text_messages = [{"role": "user", "content": "Respond with just: OK"}]
try:
    text_response, text_time = chat(text_messages)
    print(f"Answer: {text_response}")
    print(f"Time taken: {text_time:.2f} seconds")
    assert text_response.strip() == "OK", "Text test failed: expected 'OK'"
    print("Text test passed!")
except RuntimeError as e:
    print(f"Text test failed: {e}")

# --- Visual Test ---
print("\n--- Visual Test ---")
def create_red_circle_image(size=(200, 200)):
    img = Image.new('RGB', size, color = 'white')
    draw = ImageDraw.Draw(img)
    # Draw a red circle
    draw.ellipse([(size[0]*0.25, size[1]*0.25), (size[0]*0.75, size[1]*0.75)], fill='red', outline='red')

    buffered = io.BytesIO()
    img.save(buffered, format="PNG")
    return base64.b64encode(buffered.getvalue()).decode("utf-8")

red_circle_base64 = create_red_circle_image()

visual_messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Describe the image."},
            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{red_circle_base64}"}}
        ]
    }
]

try:
    visual_response, visual_time = chat(visual_messages, max_tokens=50) # Use fewer max_tokens for vision test
    print(f"Answer: {visual_response}")
    print(f"Time taken: {visual_time:.2f} seconds")
    # Basic check for vision response
    if "red circle" in visual_response.lower() or "circle" in visual_response.lower():
        print("Visual test passed (detected red circle)! ")
    else:
        print("Visual test did not detect red circle. It might be due to limited max_tokens or model specifics.")
except RuntimeError as e:
    print(f"Visual test failed: {e}")



```markdown
## 7. Public Exposure (Cloudflare Tunnel or ngrok)

This cell exposes the `llama-server` to the public internet using either Cloudflare Tunnel (preferred) or ngrok. Make sure your `TUNNEL_MODE` and secrets are configured correctly.
```

In [ ]:
import threading, time, subprocess

base_public = None
public_url_event = threading.Event()

def run_cloudflared_tunnel():
    global base_public
    print("Starting Cloudflare Tunnel...")
    # Assuming 'cloudflared' is installed via the bash cell
    cf_command = [
        "cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"
    ]
    # Cloudflare tunnel might require login or a configuration file for a persistent URL
    # For a quick public URL without a custom hostname, it usually prints a URL to stderr/stdout

    # If using a pre-configured tunnel with CF_HOSTNAME:
    # cf_command = [
    #     "cloudflared", "tunnel", "--hostname", CF_HOSTNAME, "--url", f"http://127.0.0.1:{PORT}"
    # ]

    process = subprocess.Popen(cf_command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

    while True:
        line = process.stderr.readline() # Cloudflare often prints URLs to stderr
        if line:
            print(f"[cloudflared] {line.strip()}")
            if "https://" in line and ".trycloudflare.com" in line:
                # This is for the ephemeral tunnel without custom hostname
                base_public = line.split(' ').pop().strip() + "/v1"
                print(f"Cloudflare Tunnel URL found: {base_public}")
                public_url_event.set()
                break
            elif CF_HOSTNAME and f"https://{CF_HOSTNAME}" in line:
                # For custom hostname setup, assuming it's ready after printing info
                base_public = f"https://{CF_HOSTNAME}/v1"
                print(f"Cloudflare Tunnel with custom hostname: {base_public}")
                public_url_event.set()
                break
        if process.poll() is not None:
            print("Cloudflare Tunnel process exited prematurely.")
            break
        time.sleep(1)

def run_ngrok_tunnel():
    global base_public
    if not NGROK_TOKEN:
        print("ngrok token not set, skipping ngrok tunnel.")
        return

    print("Starting ngrok tunnel...")
    from pyngrok import ngrok

    ngrok.set_auth_token(NGROK_TOKEN)
    # Disconnect any existing tunnels
    ngrok.disconnect()

    try:
        # Ensure we always get a free domain in case of rate limits
        public_url = ngrok.connect(addr=PORT, proto="http", domain=DOMAIN if DOMAIN and "ngrok-free.dev" in DOMAIN else None)
        base_public = public_url.public_url + "/v1"
        print(f"ngrok Tunnel URL found: {base_public}")
        public_url_event.set()
    except Exception as e:
        print(f"Error starting ngrok tunnel: {e}")


# Start the tunnel in a separate thread
if TUNNEL_MODE == "cloudflared":
    tunnel_thread = threading.Thread(target=run_cloudflared_tunnel, daemon=True)
elif TUNNEL_MODE == "ngrok":
    tunnel_thread = threading.Thread(target=run_ngrok_tunnel, daemon=True)
else:
    print("Invalid TUNNEL_MODE. Choose 'cloudflared' or 'ngrok'.")
    tunnel_thread = None

if tunnel_thread:
    tunnel_thread.start()
    print("Waiting for public URL...")
    public_url_event.wait(timeout=120) # Wait up to 120 seconds

if base_public:
    print(f"\nYour public API endpoint (OpenAI-compatible): {base_public}")
    print("You can use this URL to connect other applications (e.g., Hermes, OpenCode).")
    print("Example: BASE_URL = '{base_public}'")
else:
    print("Failed to get a public URL within the timeout. Check logs for errors.")

# Keep the cell alive to maintain the tunnel
try:
    while True: # Keep the tunnel alive until the cell is stopped
        time.sleep(60)
except KeyboardInterrupt:
    print("Tunnel stopped by user.")
    if 'server_process' in locals() and server_process.poll() is None:
        server_process.terminate()
    if TUNNEL_MODE == "ngrok":
        from pyngrok import ngrok
        ngrok.disconnect()
